# Week 8: Using Transformer Models

## Getting started
If working on your own machine, make sure the huggingface transformers package is installed

`conda install -c huggingface transformers`

or

`pip install transformers`

Of course, if working on Google Colab, you won't need to do this.  Whatever environment you are using check whether the following code runs.  It should output a negative label with a high score!

For `pipeline` updated version: determine the model 

In [1]:
from transformers import pipeline

'''
This pipeline is doing:

Text → Tokenizer → DistilBERT → Classification head → Softmax

Using a fine-tuned Transformer model, not just raw embeddings.
'''

sentiment = pipeline("sentiment-analysis",
                    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
                    )

result = sentiment("I hate you")
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9991129040718079}]


The first time we use a Hugging Face Transformer pipeline, the model is downloaded from the Hugging Face Hub. After that, the model is cached locally, so future runs are faster.

The following is adapted from an older version of the huggingface quickstart to transformers tutorial 
https://huggingface.co/transformers/v2.4.0/quickstart.html
We will be looking at the BERT introduction (but feel free to have a look at GPT2 etc as well!)

First of all we need some key imports.  
We are going to be using the pre-trained **bert-base-uncased** model so this cell instantiates a tokenizer for this model.  

**Logging** is also switched on so we can see more of what's going on. 

The first time you run it, the model will be downloaded and cached.  The cached version will be used on subsequent runs, if it is available (not on Google CoLab).

In [1]:
import torch
from transformers import BertTokenizer, BertModel, BertForMaskedLM

# OPTIONAL: if you want to have more information on what's happening under the hood, activate the logger as follows
import logging
logging.basicConfig(level=logging.INFO)

# Load pre-trained model tokenizer (vocabulary)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

'''
tokenized = tokenizer(
    example_sentences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)
'''





INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


'\ntokenized = tokenizer(\n    example_sentences,\n    padding=True,\n    truncation=True,\n    return_tensors="pt"\n)\n'

### 1) Now we are going to tokenize some text.  
This will demonstrate the `wordpiece` vocabulary used by BERT as well as the fact that we need to introduce special `[CLS]` and `[SEP]` tokens in the input.

In [2]:
# Tokenize input
text = "[CLS] Who was elected as British prime minister in 1951? [SEP] Sir Winston Leonard Spencer Churchill was a British politician, statesman, army officer and writer, who was Prime Minister of the United Kingdom from 1940 to 1945 and again from 1951 to 1955. [SEP]"
tokenized_text2 = tokenizer.tokenize(text)
print(tokenized_text2)

['[CLS]', 'who', 'was', 'elected', 'as', 'british', 'prime', 'minister', 'in', '1951', '?', '[SEP]', 'sir', 'winston', 'leonard', 'spencer', 'churchill', 'was', 'a', 'british', 'politician', ',', 'statesman', ',', 'army', 'officer', 'and', 'writer', ',', 'who', 'was', 'prime', 'minister', 'of', 'the', 'united', 'kingdom', 'from', '1940', 'to', '1945', 'and', 'again', 'from', '1951', 'to', '1955', '.', '[SEP]']


In [3]:
# Tokenize input
text = "[CLS] What are igneous rocks? [SEP] Igneous rocks form when hot , molten rock crystallizes and solidifies. [SEP] "
tokenized_text= tokenizer.tokenize(text)
print(tokenized_text)

['[CLS]', 'what', 'are', 'ign', '##eous', 'rocks', '?', '[SEP]', 'ign', '##eous', 'rocks', 'form', 'when', 'hot', ',', 'molten', 'rock', 'crystal', '##li', '##zes', 'and', 'solid', '##ifies', '.', '[SEP]']


In [4]:
tokenized_text[11]

'form'

Note that the tokenizer is not breaking down all words according to their morphology -- **only rare words**.  
Reasonably frequent words such as `elected` are left as whole words.  Rarer words such as `solidifies` are broken down.

### 2) Now we are going to **mask** out one of the words in the text.  

For the purposes of this demonstration, I have chosen token 11 but you could try different tokens.  

Remember that: **during training the tokens to mask are chosen randomly.**


In [5]:
# Mask a token that we will try to predict back with `BertForMaskedLM`
masked_index = 11
tokenized_text[masked_index] = '[MASK]'
print(tokenized_text)

['[CLS]', 'what', 'are', 'ign', '##eous', 'rocks', '?', '[SEP]', 'ign', '##eous', 'rocks', '[MASK]', 'when', 'hot', ',', 'molten', 'rock', 'crystal', '##li', '##zes', 'and', 'solid', '##ifies', '.', '[SEP]']


In [6]:
print(len(tokenized_text))

25


We are now going to try to use the masked language model to predict this word.

### 3) First we need to convert the input into a list of `word index ids`.

In [7]:
# Convert token to vocabulary indices
indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
print(indexed_tokens)

[101, 2054, 2024, 16270, 14769, 5749, 1029, 102, 16270, 14769, 5749, 103, 2043, 2980, 1010, 23548, 2600, 6121, 3669, 11254, 1998, 5024, 14144, 1012, 102]


### 4) We need `segment ids` to define whether a token is in the first or second sentence.

In [8]:
def make_segment_ids(list_of_tokens):

    #this function assumes that up to and including the first '[SEP]' is the first segment, anything afterwards is the second segment

    current_id=0
    segment_ids=[]
    
    for token in list_of_tokens:
        segment_ids.append(current_id)
        if token == '[SEP]':
            current_id=1
    return segment_ids


segment_ids=make_segment_ids(tokenized_text)
print(segment_ids)

[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


### 5) Convert inputs to PyTorch tensors

In [ ]:
#this just wraps things up in multi-dimensional tensors rather than as flat lists.

tokens_tensor = torch.tensor([indexed_tokens])
segments_tensors = torch.tensor([segment_ids])
print(tokens_tensor)
print(segments_tensors)

tensor([[  101,  2054,  2024, 16270, 14769,  5749,  1029,   102, 16270, 14769,
          5749,   103,  2043,  2980,  1010, 23548,  2600,  6121,  3669, 11254,
          1998,  5024, 14144,  1012,   102]])
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1]])


### 6) Now we need to encode the input using the `BertModel` `bert-base-uncased model`


In [10]:
# Load pre-trained model (weights)
model = BertModel.from_pretrained('bert-base-uncased')

# Set the model in evaluation mode to deactivate the DropOut modules
# This is IMPORTANT to have reproducible results during evaluation!
model.eval()

# If you have a GPU, put everything on cuda - otherwise comment this out to run on CPU
tokens_tensor = tokens_tensor.to('cuda')
segments_tensors = segments_tensors.to('cuda')
model.to('cuda')

# Predict hidden states features for each layer
with torch.no_grad(): # Context-manager that disables gradient calculation. Good for Inference, when not use backward

    # See the models docstrings for the detail of the inputs
    outputs = model(tokens_tensor, token_type_ids=segments_tensors)

    # Transformers models always output tuples.
    # See the models docstrings for the detail of all the outputs

    # In our case, 
    # the first element of outputs is the output of the last layer of the Bert model (all tokens)
    # the second element of outputs, outputs[1] is actually just a "pooled_output" representation of the CLS token (rather than all tokens) - however this involves an extra layer which is why it is not the same as the first element in outputs[0]!

    encoded_layers = outputs[0]
    
# We have encoded our input sequence in a FloatTensor of shape (batch size, sequence length, model hidden dimension)


INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
outputs

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.6674,  0.9458, -0.5036,  ..., -0.5302,  0.1874,  0.6336],
         [ 0.3067, -0.0335,  0.0011,  ...,  0.1267, -0.3640, -0.9253],
         [ 0.4349,  0.3408,  0.5464,  ..., -0.4371, -0.1396,  0.6499],
         ...,
         [ 0.2870,  0.5662, -0.0956,  ..., -0.5547, -0.4900, -0.2980],
         [ 0.6253,  0.0615, -0.2790,  ...,  0.0038, -0.5781, -0.4948],
         [ 0.6399,  0.0483, -0.2593,  ...,  0.0093, -0.5864, -0.4718]]],
       device='cuda:0'), pooler_output=tensor([[-0.9920, -0.8534, -0.9981,  0.9898,  0.9694, -0.7778,  0.9918,  0.7435,
         -0.9951, -1.0000, -0.9537,  0.9990,  0.9948,  0.9125,  0.9905, -0.9797,
         -0.9501, -0.9055,  0.7429, -0.9396,  0.9539,  1.0000, -0.7118,  0.8053,
          0.8925,  1.0000, -0.9784,  0.9881,  0.9908,  0.8765, -0.9697,  0.7592,
         -0.9981, -0.6847, -0.9970, -0.9994,  0.9093, -0.9470, -0.6564, -0.6737,
         -0.9759,  0.8515,  1.0000,  0.4217,  0.860

In [12]:
print(outputs.keys())

odict_keys(['last_hidden_state', 'pooler_output'])


In [13]:
''' They are same
encoded_layers = outputs[0]
encoded_layers = outputs.last_hidden_state
'''

' They are same\nencoded_layers = outputs[0]\nencoded_layers = outputs.last_hidden_state\n'

In [14]:
# We have encoded our input sequence in a FloatTensor of shape (batch size, sequence length, model hidden dimension)
# len(tokenized_text) = 25
#(1, seq_len, 768) 
# For standart BERT base models every token embedding vector is 768-dimensional.
print(encoded_layers.shape) 

torch.Size([1, 25, 768])


**BERT-Base Architecture:**

| Component          | Value |
| ------------------ | ----- |
| Hidden size        | 768   |
| Transformer layers | 12    |
| Attention heads    | 12    |


(768,)

This is a design choice.

Bigger hidden size:

- Advantage: more representational capacity
- Disadvantage: more memory/computation

| Model       | Hidden Size |
| ----------- | ----------- |
| BERT-base   | 768         |
| BERT-large  | 1024        |
| DistilBERT  | 768         |
| MiniLM      | smaller     |
| GPT-2 small | 768         |
| GPT-3       | much larger |

**768 menas 768 learned latent semantic features distributed across dimensions.**

In [15]:
text = "[CLS] What are igneous rocks? [SEP] Igneous rocks form when hot , molten rock crystallizes and solidifies. [SEP] "

In [16]:
encoded_layers[0][0].shape # [CLS] token embedding tensor

torch.Size([768])

In [17]:
encoded_layers[0][1].shape # "What" token embedding tensor

torch.Size([768])

In [18]:
#outputs[1] is a representation of the CLS token of shape (batch size, model hidden dimension)
outputs[1].shape

torch.Size([1, 768])

### 7) We can predict the masked token as follows. `BertForMaskedLM`
Using the last layer of the BERT model, we find the token id which maximises the prediction for the masked token.

Note that if you are using Google Colab you can change your runtime type between using CPU and a T4 GPU from the Runtime menu ("Change runtime type").  Whilst for the purposes if these exercises, it won't make much difference, if you are processing a lot of text, using the GPU will be much faster.  If you have access to a GPU, you need to make sure the code actually uses the GPU.  Here, you need to put everything on 'cuda' (which is essentially the GPU device) by uncommenting the lines below.  The lab machines also have quite powerful GPUs which it should also be possible to use in the same way (assuming the correct versions of various packages/drivers have been installed).

In [19]:
model = BertForMaskedLM.from_pretrained('bert-base-uncased')
model.eval()

# If you have a GPU, put everything on cuda
tokens_tensor = tokens_tensor.to('cuda')
segments_tensors = segments_tensors.to('cuda')
model.to('cuda')

# Predict all tokens
with torch.no_grad():
    outputs = model(tokens_tensor, token_type_ids=segments_tensors)
    predictions = outputs[0]

# with argmax highest scoring vocabulary word
# choose most likely predicted word ID
# find the token id which maximises the prediction for the masked token and then convert this back to a word
#.item() converts a PyTorch tensor containing ONE value into a normal Python number.
predicted_index = torch.argmax(predictions[0, masked_index]).item()

#convert token ID to list of tokens, then take first token
predicted_token = tokenizer.convert_ids_to_tokens([predicted_index])[0]
print(predicted_token)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


form


In [26]:
outputs.keys()

odict_keys(['logits'])

In [20]:
predicted_tokens = tokenizer.convert_ids_to_tokens([predicted_index])
predicted_tokens

['form']

In [21]:
#(batch, seq_len, vocab_size)
predictions.shape

torch.Size([1, 25, 30522])

In [22]:
#all vocabulary scores for that token
predictions[0, masked_index].shape

torch.Size([30522])

In [23]:
print(predicted_token)

form


`BertModel`

returns:

```
representations/embeddings
```

`BertForMaskedLM`

returns:

```
word prediction scores
```


Important Transformer Architecture Insight

`BertForMaskedLM` internally contains:

```
BertModel + prediction head

Did BERT correctly predict the masked token?

YES

### Exercise 0

#### A) Mask each token in turn and see what BERT predicts. How accurate are its predictions?  As an extension, you could look at masking multiple words in the sequence.

In [ ]:
count = 0
predicted_sentence = []
for i in range(len(tokenized_text2)):

    masked_index = i
    masked_token = tokenized_text2[masked_index]
    tokenized_text2[masked_index] = '[MASK]'
    
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text2)
    segment_ids=make_segment_ids(tokenized_text2)

    tokens_tensor = torch.tensor([indexed_tokens])
    segments_tensors = torch.tensor([segment_ids])

    model = BertForMaskedLM.from_pretrained('bert-base-uncased')
    model.eval()

    # If you have a GPU, put everything on cuda
    tokens_tensor = tokens_tensor.to('cuda')
    segments_tensors = segments_tensors.to('cuda')
    model.to('cuda')

    # Predict all tokens
    with torch.no_grad():
        outputs = model(tokens_tensor, token_type_ids=segments_tensors)
        predictions = outputs[0]

            
    # find the token id which maximises the prediction for the masked token and then convert this back to a word
    # predictions[0, masked_indexes](first batch, masked token position) , all vocabulary scores for that token
    predicted_index = torch.argmax(predictions[0, masked_index]).item()
    predicted_token = tokenizer.convert_ids_to_tokens([predicted_index])[0]
    
    if predicted_token == masked_token:
        count +=1

    predicted_sentence.append(predicted_token)
    tokenized_text2[masked_index] = masked_token
    
    print(predicted_token)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"


.


INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"


...

.


INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


the


In [26]:
print(f"Actual Tokens to Sentence: {" ".join(tokenized_text2)}")
print(f"Predicted tokens to Sentence: {" ".join(predicted_sentence)}")
print(count)
print(f"{(count / len(tokenized_text2)):.2%}")

Actual Tokens to Sentence: [CLS] who was elected as british prime minister in 1951 ? [SEP] sir winston leonard spencer churchill was a british politician , statesman , army officer and writer , who was prime minister of the united kingdom from 1940 to 1945 and again from 1951 to 1955 . [SEP]
Predicted tokens to Sentence: . he was elected as deputy prime minister in 1945 ; : sir sir churchill - , was a british diplomat , diplomat , military officer and diplomat , who was prime minister of the united kingdom from 1943 to 1945 and again from 1953 to 1953 . the
31
63.27%


#### B) Mask two tokens

In [27]:
new_tokenized_text = tokenized_text2.copy()

In [28]:
new_tokenized_text

['[CLS]',
 'who',
 'was',
 'elected',
 'as',
 'british',
 'prime',
 'minister',
 'in',
 '1951',
 '?',
 '[SEP]',
 'sir',
 'winston',
 'leonard',
 'spencer',
 'churchill',
 'was',
 'a',
 'british',
 'politician',
 ',',
 'statesman',
 ',',
 'army',
 'officer',
 'and',
 'writer',
 ',',
 'who',
 'was',
 'prime',
 'minister',
 'of',
 'the',
 'united',
 'kingdom',
 'from',
 '1940',
 'to',
 '1945',
 'and',
 'again',
 'from',
 '1951',
 'to',
 '1955',
 '.',
 '[SEP]']

In [29]:
import random
count = 0
predicted_tokens_all = []
masked_tokens_all = []

for _ in range(5):

    masked_indexes = random.sample(range(len(new_tokenized_text)), k=2)
    

    first_masked_token = new_tokenized_text[masked_indexes[0]]
    second_masked_token = new_tokenized_text[masked_indexes[1]]
    masked_tokens_all.append([first_masked_token, second_masked_token])
    
    new_tokenized_text[masked_indexes[0]] = '[MASK]'
    new_tokenized_text[masked_indexes[1]] = '[MASK]'
    
    indexed_tokens = tokenizer.convert_tokens_to_ids(new_tokenized_text)
    segment_ids=make_segment_ids(new_tokenized_text)

    tokens_tensor = torch.tensor([indexed_tokens])
    segments_tensors = torch.tensor([segment_ids])

    model = BertForMaskedLM.from_pretrained('bert-base-uncased')
    model.eval()

    # If you have a GPU, put everything on cuda
    tokens_tensor = tokens_tensor.to('cuda')
    segments_tensors = segments_tensors.to('cuda')
    model.to('cuda')

    # Predict all tokens
    with torch.no_grad():
        outputs = model(tokens_tensor, token_type_ids=segments_tensors)
        predictions_two_masked = outputs[0]

            
    # find the token id which maximises the prediction for the masked token and then convert this back to a word
    # predictions[0, masked_index] indexing: first sentence in batch and masked_index token in sequence
    #dim=0, across tokens
    #dim=1, across vocabulary
    #predictions[0, masked_indexes](first batch, masked token positions) , all vocabulary scores for that token
    #predictions_two_masked[0, masked_indexes].shape (two masked token positions, vocabulary scores)
    # two masked tokens scores for ALL vocabulary
    # when dim=1, look the entire row across columns take the max
    predicted_indexes = torch.argmax(predictions_two_masked[0, masked_indexes], dim=1)

    # .tolist() tensor object attribute to convert list
    predicted_tokens = tokenizer.convert_ids_to_tokens(predicted_indexes.tolist())
    
    if predicted_tokens[0] == first_masked_token and predicted_tokens[1] == second_masked_token:
        count +=1

    elif predicted_tokens[0] == first_masked_token or predicted_tokens[1] == second_masked_token:
        count +=0.5

    predicted_tokens_all.append(predicted_tokens)
    new_tokenized_text[masked_indexes[0]] = first_masked_token
    new_tokenized_text[masked_indexes[1]] = second_masked_token
    
    print(new_tokenized_text)
    print(predicted_tokens)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"


['[CLS]', 'who', 'was', 'elected', 'as', 'british', 'prime', 'minister', 'in', '1951', '?', '[SEP]', 'sir', 'winston', 'leonard', 'spencer', 'churchill', 'was', 'a', 'british', 'politician', ',', 'statesman', ',', 'army', 'officer', 'and', 'writer', ',', 'who', 'was', 'prime', 'minister', 'of', 'the', 'united', 'kingdom', 'from', '1940', 'to', '1945', 'and', 'again', 'from', '1951', 'to', '1955', '.', '[SEP]']
['.', '-']


INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"


...

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"


['[CLS]', 'who', 'was', 'elected', 'as', 'british', 'prime', 'minister', 'in', '1951', '?', '[SEP]', 'sir', 'winston', 'leonard', 'spencer', 'churchill', 'was', 'a', 'british', 'politician', ',', 'statesman', ',', 'army', 'officer', 'and', 'writer', ',', 'who', 'was', 'prime', 'minister', 'of', 'the', 'united', 'kingdom', 'from', '1940', 'to', '1945', 'and', 'again', 'from', '1951', 'to', '1955', '.', '[SEP]']
[',', '.']


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'who', 'was', 'elected', 'as', 'british', 'prime', 'minister', 'in', '1951', '?', '[SEP]', 'sir', 'winston', 'leonard', 'spencer', 'churchill', 'was', 'a', 'british', 'politician', ',', 'statesman', ',', 'army', 'officer', 'and', 'writer', ',', 'who', 'was', 'prime', 'minister', 'of', 'the', 'united', 'kingdom', 'from', '1940', 'to', '1945', 'and', 'again', 'from', '1951', 'to', '1955', '.', '[SEP]']
['who', 'minister']


In [35]:
outputs.keys()

odict_keys(['logits'])

In [37]:
outputs

MaskedLMOutput(loss=None, logits=tensor([[[ -6.3136,  -6.2244,  -6.3238,  ...,  -5.6784,  -5.5862,  -3.6010],
         [-11.4467, -11.2164, -11.4808,  ..., -10.2522,  -9.7345, -12.8457],
         [-12.9273, -12.6654, -12.4401,  ..., -12.9323,  -9.0732,  -8.8123],
         ...,
         [ -6.3707,  -6.2020,  -6.3940,  ...,  -6.9013,  -4.6024,  -8.8326],
         [-15.2625, -15.1074, -15.1944,  ..., -12.6047, -12.4298, -11.5565],
         [-14.8367, -14.7426, -14.7808,  ..., -12.3037, -12.2461, -11.2608]]],
       device='cuda:0'), hidden_states=None, attentions=None)

In [38]:
#(batch, seq_len, vocab_size)
predictions_two_masked.shape

torch.Size([1, 49, 30522])

In [39]:
print(predictions_two_masked[0, masked_indexes])
predictions_two_masked[0, masked_indexes].shape


tensor([[-6.3518, -6.3638, -6.2420,  ..., -5.9276, -5.1595, -7.6097],
        [-5.4341, -5.8082, -5.5597,  ..., -6.1410, -4.9256, -5.0776]],
       device='cuda:0')


torch.Size([2, 30522])

In [40]:
print(predicted_indexes)
predicted_indexes.tolist() # tensor object attribute to convert list

tensor([2040, 2704], device='cuda:0')


[2040, 2704]

In [41]:
print(count)
print(predicted_tokens_all)
print(masked_tokens_all)


1.5
[['.', '-'], ['he', 'a'], [',', '1944'], [',', '.'], ['who', 'minister']]
[['[CLS]', 'spencer'], ['who', 'a'], ['churchill', '1940'], ['churchill', '[CLS]'], ['who', 'minister']]


In [42]:
new_tokenized_text1 = tokenized_text2.copy()

In [43]:
new_tokenized_text1

['[CLS]',
 'who',
 'was',
 'elected',
 'as',
 'british',
 'prime',
 'minister',
 'in',
 '1951',
 '?',
 '[SEP]',
 'sir',
 'winston',
 'leonard',
 'spencer',
 'churchill',
 'was',
 'a',
 'british',
 'politician',
 ',',
 'statesman',
 ',',
 'army',
 'officer',
 'and',
 'writer',
 ',',
 'who',
 'was',
 'prime',
 'minister',
 'of',
 'the',
 'united',
 'kingdom',
 'from',
 '1940',
 'to',
 '1945',
 'and',
 'again',
 'from',
 '1951',
 'to',
 '1955',
 '.',
 '[SEP]']

Inside your loop:

`model = BertForMaskedLM.from_pretrained('bert-base-uncased')`
`model.eval()`
`model.to('cuda')`

This is expensive.

So each iteration:

1. Checks model
2. Loads weights
3. Moves to GPU
4. Recreates model

That is why repeatedly see logs.

I moved this OUTSIDE the loop:Then inside loop only do inference.

#### C) Mask multiple tokens

In [44]:
import random
count = 0
predicted_tokens_all = []
masked_tokens_all = []

model = BertForMaskedLM.from_pretrained('bert-base-uncased')
model.eval()
model.to('cuda')

for _ in range(5):

    masked_indexes = random.sample(range(len(new_tokenized_text1)), k=4)
    

    first_masked_token = new_tokenized_text1[masked_indexes[0]]
    second_masked_token = new_tokenized_text1[masked_indexes[1]]
    third_masked_token = new_tokenized_text1[masked_indexes[2]]
    fourth_masked_token = new_tokenized_text1[masked_indexes[3]]
    masked_tokens = [first_masked_token, second_masked_token, third_masked_token, fourth_masked_token]
    masked_tokens_all.append([masked_tokens])
    
    new_tokenized_text1[masked_indexes[0]] = '[MASK]'
    new_tokenized_text1[masked_indexes[1]] = '[MASK]'
    new_tokenized_text1[masked_indexes[2]] = '[MASK]'
    new_tokenized_text1[masked_indexes[3]] = '[MASK]'

    indexed_tokens = tokenizer.convert_tokens_to_ids(new_tokenized_text1)
    segment_ids=make_segment_ids(new_tokenized_text1)

    tokens_tensor = torch.tensor([indexed_tokens])
    segments_tensors = torch.tensor([segment_ids])


    # If you have a GPU, put everything on cuda
    tokens_tensor = tokens_tensor.to('cuda')
    segments_tensors = segments_tensors.to('cuda')


    # Predict all tokens
    with torch.no_grad():
        outputs = model(tokens_tensor, token_type_ids=segments_tensors)
        predictions = outputs[0]

            
    # find the token id which maximises the prediction for the masked token and then convert this back to a word
    # predictions[0, masked_index] indexing: first sentence in batch and masked_index token in sequence
    #dim=0, across tokens
    #dim=1, across vocabulary
    predicted_indexes = torch.argmax(predictions[0, masked_indexes], 
                                     dim=1)


    predicted_tokens = tokenizer.convert_ids_to_tokens(predicted_indexes.tolist())
    
    for token_pre, token_masked in zip(predicted_tokens, masked_tokens):
        if token_pre == token_masked:
            count +=0.25


    predicted_tokens_all.append(predicted_tokens)
    new_tokenized_text1[masked_indexes[0]] = first_masked_token
    new_tokenized_text1[masked_indexes[1]] = second_masked_token
    new_tokenized_text1[masked_indexes[2]] = third_masked_token
    new_tokenized_text1[masked_indexes[3]] = fourth_masked_token
    
    print(new_tokenized_text1)
    print(predicted_tokens)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'who', 'was', 'elected', 'as', 'british', 'prime', 'minister', 'in', '1951', '?', '[SEP]', 'sir', 'winston', 'leonard', 'spencer', 'churchill', 'was', 'a', 'british', 'politician', ',', 'statesman', ',', 'army', 'officer', 'and', 'writer', ',', 'who', 'was', 'prime', 'minister', 'of', 'the', 'united', 'kingdom', 'from', '1940', 'to', '1945', 'and', 'again', 'from', '1951', 'to', '1955', '.', '[SEP]']
['empire', 'british', 'was', 'minister']
['[CLS]', 'who', 'was', 'elected', 'as', 'british', 'prime', 'minister', 'in', '1951', '?', '[SEP]', 'sir', 'winston', 'leonard', 'spencer', 'churchill', 'was', 'a', 'british', 'politician', ',', 'statesman', ',', 'army', 'officer', 'and', 'writer', ',', 'who', 'was', 'prime', 'minister', 'of', 'the', 'united', 'kingdom', 'from', '1940', 'to', '1945', 'and', 'again', 'from', '1951', 'to', '1955', '.', '[SEP]']
['1945', '1948', 'united', 'was']
['[CLS]', 'who', 'was', 'elected', 'as', 'british', 'prime', 'minister', 'in', '1951', '?', '[SEP

In [45]:
print(count)
print(predicted_tokens_all)
print(masked_tokens_all)

3.25
[['empire', 'british', 'was', 'minister'], ['1945', '1948', 'united', 'was'], ['1945', 'sir', 'a', 'churchill'], [',', 'officer', 'british', ';'], ['was', 'from', ',', '1944']]
[[['kingdom', 'united', 'was', 'minister']], [['1940', '1945', 'united', 'was']], [['1945', 'sir', 'a', 'leonard']], [[',', 'officer', 'british', '?']], [['was', 'from', ',', '1940']]]


## Representing Sentential Meaning
We are going to be looking at different strategies for representing sentential meaning
* CLS token representation
* centroid/sum of output embeddings

The file `examples.txt` contains some example sentences.

### Exercise 1
Read in the sentences and store them as a list of sentences.  Add `[CLS]` and `[SEP]` tokens to the beginning and end of each and then pass them through the bert-base-uncased tokenizer

In [46]:
example_sentences = []

with open("examples.txt", "r") as file:
    for line in file:
        line = line.rstrip()

        sentence = "[CLS] " + line + " [SEP]"

        example_sentences.append(sentence)

example_sentences

['[CLS] The boy kicks the ball. [SEP]',
 '[CLS] The ball kicks the boy. [SEP]',
 '[CLS] The child kicks the ball. [SEP]',
 '[CLS] The ball is kicked by the boy. [SEP]',
 '[CLS] The ball is kicked. [SEP]',
 '[CLS] The boy kicks. [SEP]',
 '[CLS] The child kicks. [SEP]',
 '[CLS] The boy kicks a round object. [SEP]',
 '[CLS] The male child kicks the ball. [SEP]',
 '[CLS] The boy is playing football. [SEP]',
 '[CLS] The boy hits the ball. [SEP]',
 '[CLS] The ball hits the boy. [SEP]',
 '[CLS] The boy is hit by the ball. [SEP]',
 '[CLS] The ball is hit by the boy. [SEP]',
 '[CLS] The female child kicks the ball. [SEP]',
 '[CLS] The girl kicks the ball. [SEP]',
 '[CLS] The child plays with dolls. [SEP]',
 '[CLS] The female child plays with dolls. [SEP]',
 '[CLS] The male child plays with dolls. [SEP]',
 '[CLS] The girl plays with dolls. [SEP]',
 '[CLS] The boy plays with dolls. [SEP]',
 '[CLS] The boy is kicking the ball. [SEP]',
 '[CLS] The boy is not kicking the ball. [SEP]',
 '[CLS] All bo

In [47]:
len(example_sentences)

35

In [48]:
tokenized_sentences = [tokenizer.tokenize(sentence) for sentence in example_sentences]
tokenized_sentences

[['[CLS]', 'the', 'boy', 'kicks', 'the', 'ball', '.', '[SEP]'],
 ['[CLS]', 'the', 'ball', 'kicks', 'the', 'boy', '.', '[SEP]'],
 ['[CLS]', 'the', 'child', 'kicks', 'the', 'ball', '.', '[SEP]'],
 ['[CLS]', 'the', 'ball', 'is', 'kicked', 'by', 'the', 'boy', '.', '[SEP]'],
 ['[CLS]', 'the', 'ball', 'is', 'kicked', '.', '[SEP]'],
 ['[CLS]', 'the', 'boy', 'kicks', '.', '[SEP]'],
 ['[CLS]', 'the', 'child', 'kicks', '.', '[SEP]'],
 ['[CLS]', 'the', 'boy', 'kicks', 'a', 'round', 'object', '.', '[SEP]'],
 ['[CLS]', 'the', 'male', 'child', 'kicks', 'the', 'ball', '.', '[SEP]'],
 ['[CLS]', 'the', 'boy', 'is', 'playing', 'football', '.', '[SEP]'],
 ['[CLS]', 'the', 'boy', 'hits', 'the', 'ball', '.', '[SEP]'],
 ['[CLS]', 'the', 'ball', 'hits', 'the', 'boy', '.', '[SEP]'],
 ['[CLS]', 'the', 'boy', 'is', 'hit', 'by', 'the', 'ball', '.', '[SEP]'],
 ['[CLS]', 'the', 'ball', 'is', 'hit', 'by', 'the', 'boy', '.', '[SEP]'],
 ['[CLS]', 'the', 'female', 'child', 'kicks', 'the', 'ball', '.', '[SEP]'],
 ['[CL

In [49]:
indexed_tokens_sent = [tokenizer.convert_tokens_to_ids(sentence) for sentence in tokenized_sentences]
len(indexed_tokens_sent)

35

Because it all independent sentences i can use 0. 

Each sentence is fed separately as its own BERT input. Every Input Start With `[CLS]`

For BERT:

Typically every input sequence starts with:

`[CLS]`

and ends with:

`[SEP]`

because BERT was pretrained that way.

In [51]:
segment_ids_sent = [[0] * len(sentence) for sentence in tokenized_sentences]
print(len(segment_ids_sent))
segment_ids_sent

35


[[0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0,

In [52]:
# Padding require because different lengths

from torch.nn.utils.rnn import pad_sequence

tokens_tensor2 = pad_sequence([torch.tensor(x) for x in indexed_tokens_sent],
                                batch_first=True,
                                padding_value=tokenizer.pad_token_id
                                )

segments_tensors2 = pad_sequence([torch.tensor(x) for x in segment_ids_sent],
                                    batch_first=True,
                                    padding_value=0
                                    )

In [60]:
print(len(tokens_tensor2), tokens_tensor2.shape) # shape 35 sentence and each sentence 12 tokens
len(tokens_tensor2[0])

35 torch.Size([35, 12])


12

In [58]:
print(len(segments_tensors2), segments_tensors2.shape)
len(segments_tensors2[0])

35 torch.Size([35, 12])


12

When encoding sentences, it is actually more typical 
- to pool the hidden states for each layer (at depth n) 
- rather than the output layer.  

We can access the hidden states of the model using `output_hidden_states=True` 

In [61]:
model = BertModel.from_pretrained('bert-base-uncased')


model.eval()
tokens_tensor2 = tokens_tensor2.to('cuda')
segments_tensors2 = segments_tensors2.to('cuda')
model.to('cuda')

# Predict hidden states features for each layer
with torch.no_grad():
    # See the models docstrings for the detail of the inputs
    # output_hidden_states=True we add to output all hidden states
    outputs = model(tokens_tensor2, token_type_ids=segments_tensors2, output_hidden_states=True)

'''  
with torch.no_grad():
    outputs = model(tokens_tensor, token_type_ids=segments_tensors)
    predictions = outputs[0]
''' 
    

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'  \nwith torch.no_grad():\n    outputs = model(tokens_tensor, token_type_ids=segments_tensors)\n    predictions = outputs[0]\n'

In [62]:
print(outputs.keys())

odict_keys(['last_hidden_state', 'pooler_output', 'hidden_states'])


In [64]:
outputs.last_hidden_state.shape

torch.Size([35, 12, 768])

In [65]:
outputs.to_tuple()

(tensor([[[-0.6104,  0.0697,  0.2443,  ..., -0.0749,  0.8064,  0.6482],
          [-0.6012, -0.8302, -0.0993,  ..., -0.0835,  0.7230, -0.3246],
          [-0.9273, -0.4585,  0.4975,  ..., -0.1753,  0.4909, -0.9702],
          ...,
          [-0.4898, -0.4985,  1.0528,  ..., -0.3981, -0.0874, -0.8537],
          [-0.4123, -0.5852,  0.9165,  ..., -0.4717, -0.0216, -0.7838],
          [-0.6741, -0.2606,  0.6018,  ..., -0.4102, -0.4457, -0.7857]],
 
         [[-0.7793,  0.0874,  0.3290,  ..., -0.0565,  0.7611,  0.6928],
          [-0.6460, -0.5430,  0.0704,  ..., -0.2305,  0.3492,  0.0248],
          [-0.4873, -0.7601,  0.2256,  ..., -0.2417,  0.3207,  0.2313],
          ...,
          [-0.4670, -0.4691,  1.0474,  ..., -0.3624, -0.0559, -0.8417],
          [-0.4092, -0.5779,  0.9147,  ..., -0.4143,  0.0265, -0.7916],
          [-0.6426, -0.2214,  0.6259,  ..., -0.3488, -0.3902, -0.7690]],
 
         [[-0.6298,  0.0464,  0.2041,  ..., -0.2229,  0.7131,  0.6971],
          [-0.6854, -0.7928,

In [67]:
print(len(outputs))
for i in range(len(outputs)):
    try:
        # shapes of 'last_hidden_state', 'pooler_output'
        print(outputs[i].shape)
    except:
        # output.hidden_states is tuple does not have shape attribute
        # all hiddenstates 1 embedding layer + 12 transformers layer
        print(len(outputs[i]))

3
torch.Size([35, 12, 768])
torch.Size([35, 768])
13


Here:
* outputs[0] contains the output representation of each token
* outputs[1] is representation of the first token (after being put through an additional layer) which is CLS token
* outputs[2] is a a tuple.  Each element is the hidden layer at depth n.  If we want the last layer then we need outputs[2][-1]

THis `n` depth exactly 12transformers + 1 embedding total 13 layer in standart bert case.

In [68]:
# raw embedding representation
outputs[2][0]

# after transformer block 1
outputs[2][1]

# final contextual representation
#outputs[2][-1] is the last hidden layer also output as outputs[0]
outputs[2][-1]

tensor([[[-0.6104,  0.0697,  0.2443,  ..., -0.0749,  0.8064,  0.6482],
         [-0.6012, -0.8302, -0.0993,  ..., -0.0835,  0.7230, -0.3246],
         [-0.9273, -0.4585,  0.4975,  ..., -0.1753,  0.4909, -0.9702],
         ...,
         [-0.4898, -0.4985,  1.0528,  ..., -0.3981, -0.0874, -0.8537],
         [-0.4123, -0.5852,  0.9165,  ..., -0.4717, -0.0216, -0.7838],
         [-0.6741, -0.2606,  0.6018,  ..., -0.4102, -0.4457, -0.7857]],

        [[-0.7793,  0.0874,  0.3290,  ..., -0.0565,  0.7611,  0.6928],
         [-0.6460, -0.5430,  0.0704,  ..., -0.2305,  0.3492,  0.0248],
         [-0.4873, -0.7601,  0.2256,  ..., -0.2417,  0.3207,  0.2313],
         ...,
         [-0.4670, -0.4691,  1.0474,  ..., -0.3624, -0.0559, -0.8417],
         [-0.4092, -0.5779,  0.9147,  ..., -0.4143,  0.0265, -0.7916],
         [-0.6426, -0.2214,  0.6259,  ..., -0.3488, -0.3902, -0.7690]],

        [[-0.6298,  0.0464,  0.2041,  ..., -0.2229,  0.7131,  0.6971],
         [-0.6854, -0.7928,  0.3044,  ..., -0

In [69]:
#so if you want the penultimate hidden layer you need outputs[2][-2]
outputs[2][-2]

tensor([[[-6.3173e-01, -2.3639e-01,  1.2720e-01,  ...,  1.8155e-01,
           1.9637e-01,  6.4160e-01],
         [-8.5286e-01, -8.6454e-01,  9.0942e-02,  ...,  2.1266e-01,
           4.5258e-01, -7.8435e-02],
         [-1.0351e+00, -4.1892e-01,  6.9666e-01,  ..., -1.4403e-01,
           6.7778e-01, -1.2173e+00],
         ...,
         [-8.5043e-01, -9.2839e-01,  8.5011e-01,  ..., -2.7545e-01,
           4.8885e-02, -1.0642e+00],
         [-6.8033e-01, -1.0040e+00,  5.0214e-01,  ..., -2.5104e-01,
           2.8177e-01, -7.2317e-01],
         [-6.3980e-01, -3.8121e-01,  1.5891e-01,  ..., -2.2629e-01,
          -6.7851e-01, -9.7517e-01]],

        [[-8.7712e-01, -2.0813e-01,  1.9651e-01,  ...,  5.3511e-02,
           1.7891e-01,  7.2099e-01],
         [-6.3676e-01, -6.7474e-01,  2.5818e-01,  ..., -4.1421e-02,
           4.1413e-02,  1.7389e-01],
         [-7.4647e-01, -1.0622e+00,  2.7989e-01,  ..., -2.3324e-01,
          -1.8104e-01,  1.3757e-01],
         ...,
         [-8.1663e-01, -9

### Exercise 2

#### A) Encode each sentence using the output representation for its CLS token 

  -note that you do not need to mask the CLS token.  We are just interested in the output layer embedding for this token.  
  You can use outputs[0][0] or outputs[1] as a representation of the CLS token - 

  but you will get different results as outputs[1] as gone through an additional layer 
  (trained for next sentence prediction during fine-tuning and classification IF the model has been fine-tuned).

In [70]:
encoded_layers2 = outputs[0] # last hidden state

In [ ]:
print(encoded_layers2.shape) # [batch_size, sequence_length, hidden_size]
len(encoded_layers2) # 35 sentence representations, batch size: number of input examples processed together

torch.Size([35, 12, 768])


35

`cls_embeddings = outputs[1]` transformed CLS embedding because it is derived from CLS. CLS after additional learned projection

BUT:

`outputs[1] ≠ encoded_layers2[:,0,:]` the raw CLS embedding.

because pooler_output applies an additional learned linear layer + tanh activation to the raw CLS embedding.

In [71]:
## this is a handy way of finding the cosine similarity between two tensors
# see https://pytorch.org/docs/stable/generated/torch.nn.CosineSimilarity.html
cos = torch.nn.CosineSimilarity(dim=0, eps=1e-6)

#encoded_layers = outputs[0] contains the output representation of each token
#you use this as:
# encoded_layers2[0,0] first sentence first token = CLS
# encoded_layers2[0,1] first sentence, second token
#This computes cosine similarity between TWO TOKENS.
print(encoded_layers2[0,0],encoded_layers2[0,1])

output=cos(encoded_layers2[0,0],encoded_layers2[0,1])
print(outputs)



tensor([-6.1042e-01,  6.9678e-02,  2.4429e-01,  2.0543e-01, -4.2552e-01,
        -7.0791e-01,  1.7575e-01,  3.3443e-01,  7.5671e-02, -5.5107e-01,
        -3.9869e-01, -2.3909e-01, -1.1665e-02,  2.2262e-01,  1.7929e-01,
        -1.1460e-03, -8.0195e-02,  4.3684e-01, -2.9044e-02, -1.8289e-01,
        -3.1176e-02,  3.3740e-02,  5.8006e-01,  2.3129e-01,  2.0408e-01,
        -2.3413e-01,  6.4947e-01, -3.5945e-01, -1.7658e-01, -5.1382e-01,
        -1.1687e-01,  7.7323e-01,  1.2063e-01, -2.4284e-01,  1.1799e-01,
        -3.5865e-01, -1.4107e-01,  1.6319e-02,  1.0291e-01, -8.9070e-02,
        -1.1583e-01, -4.2834e-01,  1.8211e-01, -1.7323e-01,  8.8284e-02,
        -1.0321e+00, -4.2380e+00, -3.8747e-01, -3.6982e-01, -5.7268e-01,
        -1.3981e-01, -7.4782e-02,  7.2410e-02, -6.2640e-02,  4.7018e-02,
         5.0987e-01, -5.8288e-01, -1.2845e-02,  3.0140e-01, -2.7655e-02,
         2.2237e-01,  2.5304e-01, -2.3009e-01,  2.1233e-01, -2.6544e-01,
        -1.4457e-01, -1.2671e-01,  5.7461e-02,  9.5

* Use cosine similarity to determine all pairs similarities for the sentences.

In [72]:
len(outputs[1])

35

In [73]:
len(outputs[0][0])

12

In [75]:
print(encoded_layers2.shape)
len(encoded_layers2[0]) # encoded_layers2[0] = 

torch.Size([35, 12, 768])


12

In [77]:
cls_embeddings = encoded_layers2[:, 0, :] # raw CLS token embeddings
print(cls_embeddings.shape)

torch.Size([35, 768])


In [ ]:
sentence_similarities = {}

for i in range(len(cls_embeddings)):

    sentence_similarities[i] = {}

    for j in range(len(cls_embeddings)):

        # to ignore same tokens
        if i == j:
            continue
        
        similarity = cos(cls_embeddings[i],cls_embeddings[j])

        sentence_similarities[i][j] = similarity.item()

    sentence_similarities[i] = dict(sorted(
                                            sentence_similarities[i].items(), 
                                            key=lambda x: x[1], 
                                            reverse=True
                                           )
                                    )

sentence_similarities

{0: {10: 0.9771223664283752,
  15: 0.974289059638977,
  2: 0.972352147102356,
  1: 0.9648551940917969,
  11: 0.9572724103927612,
  9: 0.9439774751663208,
  24: 0.9362963438034058,
  7: 0.9357022047042847,
  20: 0.9256095886230469,
  5: 0.9198008179664612,
  19: 0.9138270020484924,
  16: 0.9076940417289734,
  4: 0.8992171287536621,
  6: 0.8978395462036133,
  23: 0.8975157737731934,
  8: 0.8893808126449585,
  14: 0.8857904672622681,
  32: 0.8843661546707153,
  30: 0.8771991729736328,
  31: 0.867245614528656,
  18: 0.8654366731643677,
  17: 0.8528517484664917,
  21: 0.8459711670875549,
  12: 0.8027883172035217,
  13: 0.7912527322769165,
  22: 0.7797636985778809,
  3: 0.767460823059082,
  25: 0.7586787939071655,
  26: 0.757584810256958,
  33: 0.7468137741088867,
  27: 0.7370641231536865,
  34: 0.7275315523147583,
  28: 0.7145782709121704,
  29: 0.7085939645767212},
 1: {11: 0.9829779863357544,
  2: 0.9805291295051575,
  10: 0.972331702709198,
  0: 0.9648551940917969,
  15: 0.95478457212448

#### B) Identify the 10 most similar pairs of sentences using this sentence encoding

In [79]:
all_pairs = []

for i in sentence_similarities:
    for j in sentence_similarities[i]:

        # avoid duplicate pairs
        # sentence_similarities[0][2]
        # sentence_similarities[2][0]
        if i < j:

            score = sentence_similarities[i][j]

            all_pairs.append((i, j, score))

# Sort by similarity score
all_pairs = sorted(
                    all_pairs, key=lambda x: x[2], reverse=True
                   )

top_10 = all_pairs[:10]

top_10

[(27, 34, 0.9975340962409973),
 (12, 13, 0.995598554611206),
 (17, 18, 0.994295597076416),
 (8, 14, 0.9935892820358276),
 (3, 13, 0.99186110496521),
 (5, 6, 0.9871863722801208),
 (16, 20, 0.9864925742149353),
 (16, 19, 0.9857666492462158),
 (10, 11, 0.9854278564453125),
 (3, 12, 0.9854223728179932)]

In [80]:
for i, j, score in top_10:

    print(f"Sentence {i}: {example_sentences[i]}")
    print(f"Sentence {j}: {example_sentences[j]}")
    print(f"Similarity: {score:.4f}")
    print("-" * 50)

Sentence 27: [CLS] I can see a boy kicking a ball. [SEP]
Sentence 34: [CLS] I can see a girl kicking a ball. [SEP]
Similarity: 0.9975
--------------------------------------------------
Sentence 12: [CLS] The boy is hit by the ball. [SEP]
Sentence 13: [CLS] The ball is hit by the boy. [SEP]
Similarity: 0.9956
--------------------------------------------------
Sentence 17: [CLS] The female child plays with dolls. [SEP]
Sentence 18: [CLS] The male child plays with dolls. [SEP]
Similarity: 0.9943
--------------------------------------------------
Sentence 8: [CLS] The male child kicks the ball. [SEP]
Sentence 14: [CLS] The female child kicks the ball. [SEP]
Similarity: 0.9936
--------------------------------------------------
Sentence 3: [CLS] The ball is kicked by the boy. [SEP]
Sentence 13: [CLS] The ball is hit by the boy. [SEP]
Similarity: 0.9919
--------------------------------------------------
Sentence 5: [CLS] The boy kicks. [SEP]
Sentence 6: [CLS] The child kicks. [SEP]
Similarity

### Exercise 3

#### A) Repeat exercise 2 but use the centroid of all of the output embeddings as the representation of a sentence.

Now instead of using [CLS], use the centroid / mean of token embeddings:

In [81]:
# encoded_layers2 shape torch.Size([35, 12, 768])
# mean of sentences across tokens
centroid_embeddings = encoded_layers2.mean(dim=1)
centroid_embeddings.shape

torch.Size([35, 768])

In [82]:
cos = torch.nn.CosineSimilarity(dim=0)

centroid_similarities = {}

for i in range(len(centroid_embeddings)):

    centroid_similarities[i] = {}

    for j in range(len(centroid_embeddings)):

        if i == j:
            continue

        similarity = cos(centroid_embeddings[i], centroid_embeddings[j])

        centroid_similarities[i][j] = similarity.item()

    centroid_similarities[i] = dict(sorted(
                                            centroid_similarities[i].items(),
                                            key=lambda x: x[1],
                                            reverse=True
                                            )
                                    )

#### B) Top 10 most similar pairs:

In [83]:
all_pairs = []

for i in centroid_similarities:
    for j in centroid_similarities[i]:

        if i < j:
            score = centroid_similarities[i][j]
            all_pairs.append((i, j, score))

all_pairs = sorted(all_pairs, key=lambda x: x[2], reverse=True)

top_10_centroid = all_pairs[:10]

for i, j, score in top_10_centroid:

    print(f"Sentence {i}: {example_sentences[i]}")
    print(f"Sentence {j}: {example_sentences[j]}")
    print(f"Similarity: {score:.4f}")
    print("-" * 50)

Sentence 17: [CLS] The female child plays with dolls. [SEP]
Sentence 18: [CLS] The male child plays with dolls. [SEP]
Similarity: 0.9884
--------------------------------------------------
Sentence 5: [CLS] The boy kicks. [SEP]
Sentence 6: [CLS] The child kicks. [SEP]
Similarity: 0.9871
--------------------------------------------------
Sentence 8: [CLS] The male child kicks the ball. [SEP]
Sentence 14: [CLS] The female child kicks the ball. [SEP]
Similarity: 0.9869
--------------------------------------------------
Sentence 19: [CLS] The girl plays with dolls. [SEP]
Sentence 20: [CLS] The boy plays with dolls. [SEP]
Similarity: 0.9830
--------------------------------------------------
Sentence 16: [CLS] The child plays with dolls. [SEP]
Sentence 20: [CLS] The boy plays with dolls. [SEP]
Similarity: 0.9825
--------------------------------------------------
Sentence 0: [CLS] The boy kicks the ball. [SEP]
Sentence 10: [CLS] The boy hits the ball. [SEP]
Similarity: 0.9813
-----------------

#### C) Experiment with using different pooling layers from the hidden state embeddings.  

Typically, using the penultimate layer (-2) is felt to be optimal as it is far enough away from the original uncontextualised word embeddings but also not too close to the output predictions.  
Here you should use: 'outputs.hidden_states' instead of only: 'outputs.last_hidden_state' Because hidden states contain all BERT layers.

In [87]:
hidden_states = outputs.hidden_states

print(len(hidden_states))

13


Meaning:
0  = embedding layer
1  = BERT layer 1
...
11 = BERT layer 11
12 = final BERT layer

In [86]:
hidden_states[-1]   # final layer
hidden_states[-2]   # penultimate layer
hidden_states[-3]   # third-from-last layer

tensor([[[-5.5035e-01, -5.3555e-01,  5.3248e-02,  ...,  6.3024e-03,
           3.2257e-02,  7.1278e-01],
         [-1.1031e+00, -7.0643e-01,  5.9856e-03,  ...,  1.0253e-01,
           2.5482e-01, -5.5956e-01],
         [-1.3437e+00, -4.9905e-01,  7.1719e-01,  ..., -6.5438e-01,
           6.3159e-01, -1.3360e+00],
         ...,
         [-7.8850e-01, -1.0693e+00,  4.3891e-01,  ..., -3.1613e-01,
          -1.0558e-01, -8.2707e-01],
         [-6.6886e-01, -1.1768e+00,  2.5109e-01,  ..., -2.7389e-01,
           2.2615e-01, -6.1472e-01],
         [-5.9328e-01, -3.7765e-01, -1.0598e-01,  ..., -3.0908e-01,
          -6.1562e-01, -1.0102e+00]],

        [[-6.2537e-01, -5.2487e-01,  8.5803e-02,  ..., -6.9123e-02,
           1.3769e-01,  7.3539e-01],
         [-9.3873e-01, -1.0008e+00,  1.9849e-01,  ..., -9.8599e-02,
          -1.1733e-01, -3.9493e-01],
         [-8.1578e-01, -1.2037e+00,  1.6345e-01,  ..., -2.9637e-01,
          -2.2380e-01, -3.0684e-01],
         ...,
         [-7.6504e-01, -1

In [89]:
print(hidden_states[-2].shape)

# For the penultimate layer centroid:
layer = hidden_states[-2]   # shape: [35, 12, 768]

# Centroid embedding of penultimate layer (-2)
centroid_embeddings = layer.mean(dim=1)   # shape: [35, 768]

torch.Size([35, 12, 768])


In [90]:
cos = torch.nn.CosineSimilarity(dim=0)

layer_similarities = {}

for i in range(len(centroid_embeddings)):

    layer_similarities[i] = {}

    for j in range(len(centroid_embeddings)):

        if i == j:
            continue

        score = cos(centroid_embeddings[i], centroid_embeddings[j])

        layer_similarities[i][j] = score.item()

    layer_similarities[i] = dict(sorted(layer_similarities[i].items(), key=lambda x: x[1], reverse=True))

In [91]:
# Top 10:

all_pairs = []

for i in layer_similarities:
    for j in layer_similarities[i]:

        if i < j:
            all_pairs.append(
                (i, j, layer_similarities[i][j])
            )

all_pairs = sorted(all_pairs, key=lambda x: x[2], reverse=True)

top_10_penultimate = all_pairs[:10]

for i, j, score in top_10_penultimate:

    print(f"Sentence {i}: {example_sentences[i]}")
    print(f"Sentence {j}: {example_sentences[j]}")
    print(f"Similarity: {score:.4f}")
    print("-" * 50)

Sentence 8: [CLS] The male child kicks the ball. [SEP]
Sentence 14: [CLS] The female child kicks the ball. [SEP]
Similarity: 0.9942
--------------------------------------------------
Sentence 17: [CLS] The female child plays with dolls. [SEP]
Sentence 18: [CLS] The male child plays with dolls. [SEP]
Similarity: 0.9936
--------------------------------------------------
Sentence 5: [CLS] The boy kicks. [SEP]
Sentence 6: [CLS] The child kicks. [SEP]
Similarity: 0.9911
--------------------------------------------------
Sentence 0: [CLS] The boy kicks the ball. [SEP]
Sentence 10: [CLS] The boy hits the ball. [SEP]
Similarity: 0.9899
--------------------------------------------------
Sentence 19: [CLS] The girl plays with dolls. [SEP]
Sentence 20: [CLS] The boy plays with dolls. [SEP]
Similarity: 0.9899
--------------------------------------------------
Sentence 16: [CLS] The child plays with dolls. [SEP]
Sentence 20: [CLS] The boy plays with dolls. [SEP]
Similarity: 0.9894
-----------------

In [93]:
# To experiment with different layers:

for layer_index in [-1, -2, -3, -4]:

    layer = hidden_states[layer_index]
    centroid_embeddings = layer.mean(dim=1)

    all_pairs = []

    for i in range(len(centroid_embeddings)):
        for j in range(i + 1, len(centroid_embeddings)):

            score = cos(
                centroid_embeddings[i],
                centroid_embeddings[j]
            )

            all_pairs.append((i, j, score.item()))

    all_pairs = sorted(all_pairs, key=lambda x: x[2], reverse=True)

    for i, j, score in all_pairs[:10]:
        print(f"Sentence {i}: {example_sentences[i]}")
        print(f"Sentence {j}: {example_sentences[j]}")
        print(f"Similarity: {score:.4f}")
        print("-" * 50)

Sentence 17: [CLS] The female child plays with dolls. [SEP]
Sentence 18: [CLS] The male child plays with dolls. [SEP]
Similarity: 0.9884
--------------------------------------------------
Sentence 5: [CLS] The boy kicks. [SEP]
Sentence 6: [CLS] The child kicks. [SEP]
Similarity: 0.9871
--------------------------------------------------
Sentence 8: [CLS] The male child kicks the ball. [SEP]
Sentence 14: [CLS] The female child kicks the ball. [SEP]
Similarity: 0.9869
--------------------------------------------------
Sentence 19: [CLS] The girl plays with dolls. [SEP]
Sentence 20: [CLS] The boy plays with dolls. [SEP]
Similarity: 0.9830
--------------------------------------------------
Sentence 16: [CLS] The child plays with dolls. [SEP]
Sentence 20: [CLS] The boy plays with dolls. [SEP]
Similarity: 0.9825
--------------------------------------------------
Sentence 0: [CLS] The boy kicks the ball. [SEP]
Sentence 10: [CLS] The boy hits the ball. [SEP]
Similarity: 0.9813
-----------------

I repeated the cosine similarity experiment using centroid sentence embeddings from different hidden layers. I tested the final layer (-1), penultimate layer (-2), and earlier layers (-3, -4). The penultimate layer is often preferred because it is contextualised but less specialised towards the final prediction objective than the last layer.